<a href="https://colab.research.google.com/github/nguyenngocchinh19/colab-workspace/blob/main/%E1%BB%A8ng_d%E1%BB%A5ng_AI_Camera_trong_qu%E1%BA%A3n_l%C3%BD_h%C3%A0nh_vi_%C4%91%E1%BB%91i_t%C3%A1c_v%C3%A0_gi%C3%A1m_s%C3%A1t_an_to%C3%A0n_lao_%C4%91%E1%BB%99ng_t%E1%BA%A1i_T%E1%BB%95ng_tr%E1%BA%A1m.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🎥 Trực Tiếp Thử Nghiệm Qua Webcam Laptop (Real-Time Webcam Inference)

In [ ]:
!pip install -q ultralytics
import cv2
import time
import math
import numpy as np
import threading
import queue
import requests
from ultralytics import YOLO
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

# ==============================================================================
# CẤU HÌNH FACEBOOK MESSENGER API
# ==============================================================================
FB_PAGE_ACCESS_TOKEN = "YOUR_FACEBOOK_PAGE_ACCESS_TOKEN"
FB_RECIPIENT_PSID = "YOUR_RECIPIENT_PSID"

def send_messenger_alert(image, message_text):
    """
    Hàm gửi tin nhắn cảnh báo văn bản và ảnh tới Facebook Messenger sử dụng Facebook Graph API.
    """
    if FB_PAGE_ACCESS_TOKEN == "YOUR_FACEBOOK_PAGE_ACCESS_TOKEN":
        return
    def run_send():
        try:
            url = f"https://graph.facebook.com/v18.0/me/messages?access_token={FB_PAGE_ACCESS_TOKEN}"
            payload = {
                "recipient": {"id": FB_RECIPIENT_PSID},
                "message": {"text": message_text}
            }
            headers = {"Content-Type": "application/json"}
            requests.post(url, json=payload, headers=headers, timeout=5)

            _, img_encoded = cv2.imencode('.jpg', image)
            img_bytes = img_encoded.tobytes()
            files = {
                'filedata': ('alert.jpg', img_bytes, 'image/jpeg')
            }
            data = {
                'recipient': '{"id":"' + FB_RECIPIENT_PSID + '"}',
                'message': '{"attachment":{"type":"image", "payload":{}}}'
            }
            requests.post(url, data=data, files=files, timeout=8)
        except Exception as e:
            print(f"\n[Messenger Alert] Lỗi khi gửi tới Messenger: {e}")

    threading.Thread(target=run_send, daemon=True).start()

# ==============================================================================
# 1. KHỞI TẠO MÔ HÌNH HÌNH ẢNH (CHỈ NẠP NẾU CHƯA CÓ TRONG BỘ NHỚ KERNEL)
# ==============================================================================
if 'model' not in globals():
    print("Đang nạp mô hình YOLOv8-Pose mới...")
    model = YOLO('yolov8n-pose.pt')
else:
    print("🔄 Đã tìm thấy mô hình YOLOv8-Pose trong bộ nhớ, bỏ qua bước nạp lại từ đầu!")

input_queue = queue.Queue(maxsize=1)
output_queue = queue.Queue(maxsize=1)
running = True

# ==============================================================================
# 2. HÀM THUẬT TOÁN XỬ LÝ HÌNH ẢNH & NHẬN DIỆN NGÃ
# ==============================================================================
def is_point_in_polygon(point, polygon):
    return cv2.pointPolygonTest(polygon, point, False) >= 0

def check_fall_by_skeleton(keypoints):
    if len(keypoints) < 13:
        return False
    nose = keypoints[0]
    left_shoulder = keypoints[5]
    right_shoulder = keypoints[6]
    left_hip = keypoints[11]
    right_hip = keypoints[12]

    if left_shoulder[0] > 0 and right_shoulder[0] > 0 and left_hip[0] > 0 and right_hip[0] > 0:
        shoulder_y = (left_shoulder[1] + right_shoulder[1]) / 2.0
        hip_y = (left_hip[1] + right_hip[1]) / 2.0
        shoulder_x = (left_shoulder[0] + right_shoulder[0]) / 2.0
        hip_x = (left_hip[0] + right_hip[0]) / 2.0

        dx = abs(shoulder_x - hip_x)
        dy = abs(shoulder_y - hip_y)
        angle = math.degrees(math.atan2(dx, dy + 1e-5))

        # Tiêu chí 1: Thân người nghiêng quá 60 độ
        if angle > 60:
            return True

        # Tiêu chí 2: Đầu (mũi) hạ thấp xuống ngang hoặc thấp hơn hông
        if nose[1] > 0 and nose[1] > (hip_y - 15):
            return True

    return False

# ==============================================================================
# 3. LUỒNG XỬ LÝ AI CHẠY NGẦM (BACKGROUND AI THREAD)
# ==============================================================================
def ai_inference_worker():
    global running
    polygon_points = np.array([
        [0, 0],
        [320, 0],
        [320, 240],
        [0, 240]
    ], np.int32)
    while running:
        try:
            img = input_queue.get(timeout=1)
        except queue.Empty:
            continue
        small_img = cv2.resize(img, (320, 240))
        results = model(small_img, verbose=False)
        has_violation = False
        has_fall = False
        detections = []
        for r in results:
            boxes = r.boxes.xyxy.cpu().numpy() * 2.0
            keypoints_list = (r.keypoints.xy.cpu().numpy() * 2.0) if r.keypoints is not None else []
            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = box[:4]
                center_point = (int((x1 + x2) / 2), int((y1 + y2) / 2))
                in_zone = is_point_in_polygon(center_point, polygon_points)
                if in_zone:
                    has_violation = True

                kpts = keypoints_list[i] if i < len(keypoints_list) else []
                is_falling = check_fall_by_skeleton(kpts)
                if is_falling:
                    has_fall = True

                detections.append({
                    'box': [int(x1), int(y1), int(x2), int(y2)],
                    'is_falling': is_falling,
                    'in_zone': in_zone,
                    'keypoints': kpts
                })
        if output_queue.full():
            try:
                output_queue.get_nowait()
            except queue.Empty:
                pass
        output_queue.put((detections, has_violation, has_fall))

ai_thread = threading.Thread(target=ai_inference_worker, daemon=True)
ai_thread.start()

# ==============================================================================
# 4. JAVASCRIPT KẾT NỐI CAMERA & ÂM THANH
# ==============================================================================
def start_webcam_stream():
    js = Javascript('''
        var video;
        var container_div;
        var canvas = null;
        var ctx = null;
        var stream;
        var stopStream = false;
        var lastAlert = "none";
        var lastAlertTime = 0;

        function speakVietnameseSpeech(text) {
            try {
                window.speechSynthesis.cancel();
                var msg = new SpeechSynthesisUtterance();
                msg.text = text;
                var voices = window.speechSynthesis.getVoices();
                var viVoice = voices.find(function(v) {
                    return v.lang.indexOf(\'vi\') !== -1 || v.lang.indexOf(\'VI\') !== -1;
                });
                if (viVoice) {
                    msg.voice = viVoice;
                    msg.lang = viVoice.lang;
                } else {
                    msg.lang = \'vi-VN\';
                }
                msg.rate = 1.4;
                msg.volume = 1.0;
                window.speechSynthesis.speak(msg);
            } catch(e) {
                console.log("Lỗi phát giọng nói: " + e);
            }
        }

        async function initWebcam() {
            container_div = document.createElement("div");
            container_div.style.cssText = "display: flex; flex-direction: column; align-items: center;";
            video = document.createElement("video");
            video.style.display = "block";
            video.width = 640;
            video.height = 480;
            video.autoplay = true;
            video.playsInline = true;

            const out_img = document.createElement("img");
            out_img.style.display = "block";
            out_img.width = 640;
            out_img.height = 480;

            const active_btn = document.createElement("button");
            active_btn.textContent = "🔊 Mở khóa âm thanh & Kiểm tra giọng nói";
            active_btn.style.cssText = "margin-top: 10px; margin-bottom: 10px; padding: 12px; background-color: #4CAF50; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 15px; font-weight: bold; width: 100%; max-width: 640px;";
            active_btn.onclick = () => {
                speakVietnameseSpeech("Hệ thống âm thanh sẵn sàng!");
                active_btn.style.backgroundColor = "#008CBA";
                active_btn.textContent = "🔊 Âm thanh hoạt động ổn định!";
            };

            const button = document.createElement("button");
            button.textContent = "Dừng Giám Sát";
            button.style.cssText = "margin-top: 15px; padding: 10px 20px; background-color: #f44336; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 16px;";

            container_div.appendChild(active_btn);
            container_div.appendChild(video);
            container_div.appendChild(out_img);
            container_div.appendChild(button);
            document.body.appendChild(container_div);

            try {
                stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
                video.srcObject = stream;
                await video.play();
            } catch (err) {
                alert("Không mở được camera: " + err.message);
                return;
            }

            canvas = document.createElement("canvas");
            canvas.width = 640;
            canvas.height = 480;
            ctx = canvas.getContext("2d");
            video.style.display = "none";

            button.onclick = () => {
                stopStream = true;
                window.speechSynthesis.cancel();
                stream.getTracks().forEach(track => track.stop());
                container_div.remove();
            };
        }

        async function captureAndRender(python_processed_base64, alertType) {
            if (stopStream) return null;
            if (python_processed_base64) {
                const img_elem = container_div.querySelector("img");
                if (img_elem) {
                    img_elem.src = python_processed_base64;
                }
            }
            var now = Date.now();
            if (alertType !== "none") {
                if (alertType !== lastAlert || (now - lastAlertTime > 2500)) {
                    if (alertType === "violation") {
                        speakVietnameseSpeech("Phát hiện xâm nhập vùng cấm!");
                    } else if (alertType === "fall") {
                        speakVietnameseSpeech("Cảnh báo nguy hiểm! Có người ngã!");
                    }
                    lastAlert = alertType;
                    lastAlertTime = now;
                }
            } else {
                lastAlert = "none";
            }

            if (!video || !ctx || video.readyState < 2) {
                await new Promise(resolve => window.requestAnimationFrame(resolve));
                return "WAIT";
            }
            try {
                ctx.drawImage(video, 0, 0, 640, 480);
                return canvas.toDataURL("image/jpeg", 0.6);
            } catch (e) {
                console.log("Lỗi vẽ hình: " + e);
                return "WAIT";
            }
        }

        window.initWebcam = initWebcam;
        window.captureAndRender = captureAndRender;
    ''')
    display(js)

# ==============================================================================
# 5. VÒNG LẶP LIÊN KẾT LUỒNG CHÍNH (VẼ MÔ PHỎNG KHUNG XƯƠNG HOÀN CHỈNH)
# ==============================================================================
start_webcam_stream()
eval_js('initWebcam()')

polygon_points = np.array([
    [0, 0],
    [320, 0],
    [320, 240],
    [0, 240]
], np.int32)

last_processed_img = ""
alert_type_to_js = "none"
current_detections = []
last_violation = False
last_fall = False

last_fb_send_time = 0
fb_send_cooldown = 15

print("⚡ Đã khởi động luồng giám sát kèm mô phỏng xương khớp trực quan...")

try:
    while True:
        js_reply = eval_js(f'captureAndRender("{last_processed_img}", "{alert_type_to_js}")')
        if not js_reply:
            break
        if js_reply == "WAIT":
            time.sleep(0.01)
            continue

        image_bytes = b64decode(js_reply.split(',')[1])
        jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
        img = cv2.imdecode(jpg_as_np, flags=1)

        if input_queue.empty():
            input_queue.put(img)

        if not output_queue.empty():
            current_detections, last_violation, last_fall = output_queue.get()

        zone_color = (0, 0, 255) if last_violation else (0, 255, 0)
        cv2.polylines(img, [polygon_points], isClosed=True, color=zone_color, thickness=2)

        for det in current_detections:
            x1, y1, x2, y2 = det['box']
            kpts = det.get('keypoints', [])

            box_color = (0, 0, 255) if (det['is_falling'] or det['in_zone']) else (0, 255, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), box_color, 2)

            if det['in_zone']:
                cv2.putText(img, "XAM NHAP!", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            if det['is_falling']:
                cv2.putText(img, "CO NGUOI NGA!", (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            # ==========================================================
            # VẼ MÔ PHỎNG KHUNG XƯƠNG HOÀN CHỈNH (SKELETON CONNECTIONS)
            # ==========================================================
            if len(kpts) >= 17:
                skeleton_edges = [
                    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12),
                    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
                ]

                for edge in skeleton_edges:
                    pt1_idx, pt2_idx = edge
                    pt1 = kpts[pt1_idx]
                    pt2 = kpts[pt2_idx]
                    if pt1[0] > 0 and pt1[1] > 0 and pt2[0] > 0 and pt2[1] > 0:
                        cv2.line(img, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), (255, 255, 0), 2)

                for pt in kpts:
                    if pt[0] > 0 and pt[1] > 0:
                        cv2.circle(img, (int(pt[0]), int(pt[1])), 4, (0, 0, 255), -1)

                left_shoulder = kpts[5]
                right_shoulder = kpts[6]
                left_hip = kpts[11]
                right_hip = kpts[12]
                if left_shoulder[0] > 0 and right_shoulder[0] > 0 and left_hip[0] > 0 and right_hip[0] > 0:
                    shoulder_center = (int((left_shoulder[0] + right_shoulder[0]) / 2.0),
                                       int((left_shoulder[1] + right_shoulder[1]) / 2.0))
                    hip_center = (int((left_hip[0] + right_hip[0]) / 2.0),
                                  int((left_hip[1] + right_hip[1]) / 2.0))
                    cv2.line(img, shoulder_center, hip_center, (0, 255, 255), 3)

        # Xử lý kích hoạt gửi thông báo
        current_time = time.time()
        if last_fall:
            alert_type_to_js = "fall"
            cv2.putText(img, "FALL ALARM ACTIVE", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            if current_time - last_fb_send_time > fb_send_cooldown:
                send_messenger_alert(img, "[Cảnh báo nguy hiểm] Phát hiện có người bị ngã từ hệ thống giám sát camera!")
                last_fb_send_time = current_time
        elif last_violation:
            alert_type_to_js = "violation"
            cv2.putText(img, "ZONE ALARM ACTIVE", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            if current_time - last_fb_send_time > fb_send_cooldown:
                send_messenger_alert(img, "[Cảnh báo bảo mật] Phát hiện có sự xâm nhập vào vùng cấm ảo!")
                last_fb_send_time = current_time
        else:
            alert_type_to_js = "none"

        _, encoded_img = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), 60])
        base64_str = b64encode(encoded_img).decode('utf-8')
        last_processed_img = f"data:image/jpeg;base64,{base64_str}"

except Exception as e:
    print(f"ℹ️ Đã dừng luồng camera: {e}")
finally:
    running = False

🔄 Đã tìm thấy mô hình YOLOv8-Pose trong bộ nhớ, bỏ qua bước nạp lại từ đầu!


<IPython.core.display.Javascript object>

⚡ Đã khởi động luồng giám sát kèm mô phỏng xương khớp trực quan...
